# AI101 Assignment 4 - Task 2 Used Car Dataset

**Purpose:** Repeat the prompt-based AI analysis using a different dataset: used-car features and prices.



**Model note:** The assignment's original `gemini-2.0-flash-lite` endpoint was shut down on June 1, 2026. This notebook uses `gemini-3.5-flash-lite` with Google's current `google-genai` SDK.

**Dataset note:** This project uses a small synthetic dataset created for practice. It is not real financial or vehicle-market data. The AI-generated predictions are demonstrations, not professional advice.

## 1. Install and import libraries
The first cell installs Google's current Gemini Python SDK (`google-genai`). The remaining imports handle tables, files, and graphs.

In [ ]:
!pip -q install google-genai pandas matplotlib

In [ ]:
import os
import re
import io
import getpass
import pandas as pd
import matplotlib.pyplot as plt
from google import genai

# In Colab, add GEMINI_API_KEY under the Secrets panel to avoid retyping it.
# If no Colab secret or environment variable is set, prompt privately.
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"

# Quick connection check requested by the assignment handout.
demo_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="What is the capital of France?"
)
print(demo_response.text)

## 2. Load the CSV dataset
In Colab, upload only `used_car_prices.csv` from the repository's `data` folder when prompted. The notebook stops with a clear error if a different file is selected, then reads the CSV into a pandas DataFrame. A DataFrame is a table that Python can analyze.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select only used_car_prices.csv.
file_path = "used_car_prices.csv"

# Stop clearly if the expected dataset was not uploaded.
if file_path not in uploaded:
    raise FileNotFoundError("Upload used_car_prices.csv from the repository's data folder.")

df = pd.read_csv(file_path)
print("Dataset preview:")
display(df.head())
print("\nDataset dimensions:", df.shape)
print("\nColumn information:")
df.info()

## 3. Ask Gemini to analyze the data
This prompt gives Gemini the entire CSV and asks for trends, metrics, and observations. The output is natural-language analysis rather than a prediction table. The full dataset, including its target column, is included for analysis only.

In [ ]:
csv_string = df.to_csv(index=False)

analysis_prompt = f'''
Analyze the following dataset in CSV format.
Explain the main trends, important metrics, relationships between
columns, and any unusual observations. Use specific examples from
the data. Do not claim that correlation proves causation.
State that the dataset is synthetic.

DATA:
{csv_string}
'''

analysis_response = client.models.generate_content(
    model=MODEL_NAME, contents=analysis_prompt
)
print(analysis_response.text)

## 4. Ask Gemini to predict the target value
The next prompt tells Gemini which column is the target and which column identifies each row. The actual `price_usd` column is removed before this request, so Gemini cannot see the answer it is asked to predict. It requests machine-readable CSV output so it can be loaded into pandas.

In [ ]:
# Exclude the actual target from model input to prevent target leakage.
prediction_csv = df.drop(columns=["price_usd"]).to_csv(index=False)

prediction_prompt = f'''
You are analyzing a synthetic practice dataset.
Predict the value of the "price_usd" column for every row using
the other available columns. Do not use the actual "price_usd"
value as an input when making the prediction.
Keep each row matched to its original "car_id".
Return exactly two columns named "car_id" and "predicted_price_usd".
Return one prediction for every input row.
Output ONLY valid CSV data: no explanation, no Markdown fences.

DATA:
{prediction_csv}
'''

prediction_response = client.models.generate_content(
    model=MODEL_NAME, contents=prediction_prompt
)
raw_prediction_text = prediction_response.text
print(raw_prediction_text)

## 5. Convert Gemini's CSV response into a DataFrame
This code removes optional Markdown fences, skips any leading explanation, and parses the CSV header and rows with pandas. It normalizes the expected car_id key and prediction column, converts the prediction to a number, and checks for duplicate, missing, or extra identifiers. A validated one-to-one merge then attaches predictions to the original rows. If Gemini omits or duplicates a row, the notebook stops with a clear error instead of silently reporting incomplete results.

In [ ]:
def response_to_dataframe(response_text):
    # Remove Markdown code fences if Gemini includes them.
    cleaned = response_text.strip()
    cleaned = re.sub(r"^```(?:csv)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Keep the response starting at the expected CSV header.
    lines = cleaned.splitlines()
    header_index = None
    for i, line in enumerate(lines):
        if "car_id" in line and "predicted_" in line:
            header_index = i
            break
    if header_index is None:
        raise ValueError("Could not find the expected CSV header in Gemini's response.")

    csv_text = "\n".join(lines[header_index:])
    return pd.read_csv(io.StringIO(csv_text))

predictions = response_to_dataframe(raw_prediction_text)
predictions.columns = predictions.columns.str.strip()
expected_prediction_column = "predicted_price_usd"

# Match the identifier and prediction columns even if capitalization varies.
id_match = next((c for c in predictions.columns if c.lower() == "car_id".lower()), None)
pred_match = next((c for c in predictions.columns if c.lower() == expected_prediction_column.lower()), None)
if id_match is None or pred_match is None:
    raise ValueError(f"Expected columns: car_id and {expected_prediction_column}")

predictions = predictions[[id_match, pred_match]].rename(
    columns={id_match: "car_id", pred_match: expected_prediction_column}
)
predictions[expected_prediction_column] = pd.to_numeric(
    predictions[expected_prediction_column], errors="coerce"
)
predictions = predictions.dropna(subset=[expected_prediction_column])

# Check that Gemini returned exactly one numeric prediction for each source row.
if predictions["car_id"].duplicated().any():
    raise ValueError("Gemini returned duplicate row identifiers.")
missing_ids = set(df["car_id"]) - set(predictions["car_id"])
extra_ids = set(predictions["car_id"]) - set(df["car_id"])
if missing_ids or extra_ids or len(predictions) != len(df):
    raise ValueError(f"Prediction rows do not match the source data. Missing: {missing_ids}; extra: {extra_ids}")

# Join predictions to the original data using the row identifier.
results = df.merge(predictions, on="car_id", how="left", validate="one_to_one")
if results["predicted_price_usd"].isna().any():
    raise ValueError("Some source rows do not have a valid prediction.")
display(results.head())
print("Rows with predictions:", len(results))

## 6. Compare actual values with AI predictions
The table compares the original target column with Gemini's predicted column. The mean absolute error (MAE) is the average absolute difference between actual and predicted values. Lower MAE means closer predictions on this dataset.

In [ ]:
results["absolute_error"] = (
    results["price_usd"] - results["predicted_price_usd"]
).abs()

print("Mean absolute error:", round(results["absolute_error"].mean(), 2))
display(results[["car_id", "price_usd", "predicted_price_usd", "absolute_error"]])

## 7. Plot actual versus predicted values
The graph uses the row identifier on the horizontal axis and the actual and predicted values on the vertical axis. This makes it easier to see where the AI predictions are above or below the observed values.

In [ ]:
plot_df = results.reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(plot_df["car_id"].astype(str), plot_df["price_usd"],
         marker="o", label="Actual price_usd")
plt.plot(plot_df["car_id"].astype(str), plot_df["predicted_price_usd"],
         marker="x", label="AI predicted price_usd")
plt.xticks(rotation=60)
plt.xlabel("car_id")
plt.ylabel("price_usd")
plt.title("Actual vs. AI-Predicted price_usd")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Save the results
This final cell saves the original data plus the AI predictions and errors as a new CSV file. In Colab, it also downloads the file to your computer.

In [ ]:
output_file = "ai_prediction_results.csv"
results.to_csv(output_file, index=False)
print("Saved:", output_file)

from google.colab import files
files.download(output_file)

## Prompts used
1. **Data analysis prompt:** asks Gemini to explain trends, metrics, relationships, and unusual observations in the CSV.
2. **Prediction prompt:** asks Gemini to predict the target column for every row, preserve the car_id row key, and return only CSV with the identifier and predicted value.

## Important limitation
The model's predictions can vary between runs and may be inaccurate. This notebook evaluates predictions against the same rows supplied to the model; it is a demonstration of prompt-based analysis, not a rigorous test of predictive performance on unseen data. A stronger evaluation would reserve a separate test set.